In [1]:
!pip install tradingview-screener pandas yfinance

import pandas as pd
from tradingview_screener import stocks
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor

# グローバルスコープで変数を初期化
df_output = pd.DataFrame()

# 1. 取得するカラムの定義
required_columns = [
    "name",                          # 銘柄コード
    "description",                   # 銘柄名 (デフォルトは英語)
    "close",                         # 現在値
    "change",                        # 前日比率(%)
    "market_cap_basic",              # 時価総額
    "price_earnings_ttm",            # PER
    "price_book_fq",                 # PBR
    "return_on_equity_fq",           # ROE (%)
    "dividends_yield_current",       # 配当利回り (%)
    "dividend_payout_ratio_fy",      # 配当性向 (FY)
    "ebitda_yoy_growth_fy",          # EBITDA成長率通期 (%)
    "current_ratio_fq",              # 流動比率 (%)
    "debt_to_equity_fq",             # D/Eレシオ (%)
    "total_assets_fq",               # 総資産 (自己資本比率計算用)
    "total_liabilities_fq"           # 総負債 (自己資本比率計算用)
]

print("TradingViewからデータを取得中...")
try:
    count, df_raw = (stocks('japan')
                     .select(*required_columns)
                     .limit(4000)
                     .get_scanner_data())

    # 自己資本比率の計算 (%)
    df = df_raw.copy()
    df["equity_ratio_calculated"] = (
        (df["total_assets_fq"] - df["total_liabilities_fq"]) / df["total_assets_fq"]
    ) * 100

    # スクリーニング条件の設定
    df_filtered = df[
        (df["price_earnings_ttm"] <= 20) &
        (df["price_book_fq"] <= 1.2) &
        (df["return_on_equity_fq"] >= 7.5) &
        (df["dividends_yield_current"] >= 3.8) &
        (df["ebitda_yoy_growth_fy"] >= -6) &
        (df["equity_ratio_calculated"] >= 48)
    ].copy()

    # 銘柄名を日本語名に置き換える処理
    print(f"スクリーニング対象 {len(df_filtered)} 件の日本語銘柄名を取得中...")

    def get_japanese_name(ticker_code):
        try:
            # 日本株のコードに .T を付与してYahoo Financeから取得
            t = yf.Ticker(f"{ticker_code}.T")
            info = t.info
            # longName や shortName から日本語名を取得（無ければ英語の description を使う）
            return info.get('longName') or info.get('shortName') or None
        except:
            return None

    # 並列処理で高速に日本語名を取得
    tickers = df_filtered['name'].tolist()
    with ThreadPoolExecutor(max_workers=10) as executor:
        japanese_names = list(executor.map(get_japanese_name, tickers))

    # 日本語名が取得できた場合は置き換え、取得できなかった場合は元の description を維持
    df_filtered['japanese_name'] = [
        jp_name if jp_name else eng_name
        for jp_name, eng_name in zip(japanese_names, df_filtered['description'])
    ]

    # 表示データの整形
    df_filtered["market_cap_basic"] = df_filtered["market_cap_basic"] / 100_000_000  # 億円単位化
    df_filtered = df_filtered.sort_values(by="market_cap_basic", ascending=False)      # 時価総額順

    rename_dict = {
        "name": "コード",
        "japanese_name": "銘柄名",
        "close": "現在値",
        "change": "前日比(%)",
        "market_cap_basic": "時価総額(億円)",
        "price_earnings_ttm": "PER",
        "price_book_fq": "PBR",
        "return_on_equity_fq": "ROE(%)",
        "dividends_yield_current": "配当利回り(％)",
        "dividend_payout_ratio_fy": "配当性向_FY(%)",
        "ebitda_yoy_growth_fy": "EBITDA成長率_FY(%)",
        "current_ratio_fq": "流動比率(%)",
        "debt_to_equity_fq": "D/Eレシオ(%)",
        "equity_ratio_calculated": "自己資本比率(%)"
    }

    df_output_temp = df_filtered.rename(columns=rename_dict)

    column_order = [
        "コード",
        "銘柄名",
        "現在値",
        "前日比(%)",
        "時価総額(億円)",
        "配当利回り(％)",
        "配当性向_FY(%)",
        "PER",
        "PBR",
        "ROE(%)",
        "EBITDA成長率_FY(%)",
        "自己資本比率(%)",
        "D/Eレシオ(%)",
        "流動比率(%)"
    ]
    df_output = df_output_temp[column_order]

    print(f"スクリーニング完了: {len(df_output)}件 / 全{count}件")

except Exception as e:
    print("エラーが発生しました:")
    print(e)

TradingViewからデータを取得中...
スクリーニング対象 115 件の日本語銘柄名を取得中...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 2902.T"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 5579.T"}}}
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: 242A.T"}}}


スクリーニング完了: 115件 / 全3867件


In [2]:
import pandas as pd
from IPython.display import display, HTML

# 表示用に数値を小数点以下2桁に丸める
df_display = df_output.copy()
float_cols = df_display.select_dtypes(include=['float']).columns
df_display[float_cols] = df_display[float_cols].round(2)

# CSSスタイルを定義 (ヘッダー項目名のみをダークモード対応で高コントラストに指定)
style = """
<style>
.scroll-container {
    max-height: 500px;
    overflow-y: auto;
    border: 1px solid var(--colab-border-color, #ddd);
}
.scroll-table {
    border-collapse: collapse;
    width: 100%;
    font-family: sans-serif;
}
.scroll-table th, .scroll-table td {
    padding: 8px 12px;
    border: 1px solid var(--colab-border-color, #ddd);
    text-align: left;
}
/* 項目名（ヘッダー）のみのスタイル調整 */
.scroll-table th {
    position: sticky;
    top: 0;
    background-color: #333333; /* ヘッダー背景を暗い色に固定 */
    color: #ffffff;            /* 文字色を白に固定して高い視認性を確保 */
    font-weight: bold;
    z-index: 1;
    cursor: pointer;
}
.scroll-table tr:hover {background-color: var(--colab-action-hover-color, rgba(128, 128, 128, 0.1));}
</style>
"""

# テーブルを動的にソートできるようにするための簡易JavaScript
js_script = """
<script>
function sortTable(tableId, colIndex, isNum) {
    var table = document.getElementById(tableId);
    var rows = Array.from(table.rows).slice(1);
    var asc = table.getAttribute('data-asc') === 'true';

    rows.sort(function(a, b) {
        var valA = a.cells[colIndex].innerText;
        var valB = b.cells[colIndex].innerText;
        if (isNum) {
            return asc ? parseFloat(valA) - parseFloat(valB) : parseFloat(valB) - parseFloat(valA);
        } else {
            return asc ? valA.localeCompare(valB) : valB.localeCompare(valA);
        }
    });

    table.setAttribute('data-asc', !asc);
    rows.forEach(row => table.appendChild(row));
}
</script>
"""

# HTMLテーブルの組み立て
table_id = "stock_table"
html_table = f'<div class="scroll-container">'
html_table += f'<table id="{table_id}" class="scroll-table" data-asc="true">'

# ヘッダー作成 (クリックでソート)
html_table += '<thead><tr>'
for i, col in enumerate(df_display.columns):
    is_numeric = "true" if col in float_cols or col == "コード" else "false"
    html_table += f'<th onclick="sortTable(\'{table_id}\', {i}, {is_numeric})">{col} ↕</th>'
html_table += '</tr></thead>'

# データ行作成
html_table += '<tbody>'
for _, row in df_display.iterrows():
    html_table += '<tr>'
    for val in row:
        html_table += f'<td>{val}</td>'
    html_table += '</tr>'
html_table += '</tbody></table></div>'

# 表示
display(HTML(style + js_script + html_table))

コード ↕,銘柄名 ↕,現在値 ↕,前日比(%) ↕,時価総額(億円) ↕,配当利回り(％) ↕,配当性向_FY(%) ↕,PER ↕,PBR ↕,ROE(%) ↕,EBITDA成長率_FY(%) ↕,自己資本比率(%) ↕,D/Eレシオ(%) ↕,流動比率(%) ↕
6902,DENSO Corporation,1862.0,-0.53,51473.18,3.85,41.11,11.58,0.91,8.71,9.69,60.41,0.26,2.12
8572,"Acom Co., Ltd.",451.1,0.94,7112.43,5.07,43.28,10.93,0.99,9.34,63.85,48.57,1.04,3.84
5110,"Sumitomo Rubber Industries, Ltd.",1987.0,-0.28,5336.91,4.14,40.18,8.44,0.69,8.87,-0.97,51.23,0.46,1.77
641A,NOK Group Corporation,2757.0,0.51,5286.57,4.82,45.64,8.99,0.69,8.2,-2.56,70.49,0.1,2.19
4540,Tsumura & Co.,3801.0,-0.31,2921.06,4.14,39.07,9.37,0.87,9.81,-5.88,62.92,0.45,3.55
6995,"Tokai Rika Co., Ltd.",3090.0,0.49,2701.08,3.83,33.21,9.13,0.83,8.77,5.17,67.53,0.03,2.28
7994,Okamura Corporation,2242.0,-0.31,2215.62,4.67,43.92,10.25,1.04,10.66,7.91,69.97,0.15,2.91
7278,EXEDY Corporation,5680.0,0.18,2099.88,5.65,80.15,14.45,1.08,7.82,0.32,64.26,0.28,2.48
7296,"F.C.C. Co., Ltd.",4165.0,-0.83,2003.65,5.14,50.08,10.14,0.97,10.23,0.04,78.15,0.03,3.48
4041,"Nippon Soda Co., Ltd.",3605.0,0.0,1983.6,4.66,47.54,10.61,0.91,9.21,-3.25,66.38,0.26,2.34
